In [52]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="students-group3")

# Movies encodés
query_movies = """
SELECT *
FROM `students-group3.MovieData.movies_encoded`
"""
df_movies = client.query(query_movies).to_dataframe()

# Ratings
query_ratings = """
SELECT *
FROM `students-group3.MovieData.ratings_cleaned`
"""
df_ratings = client.query(query_ratings).to_dataframe()

print(df_movies.shape)
print(df_ratings.shape)


(10329, 22)
(105339, 3)


## Matrice utilisateur–film

In [4]:
import pandas as pd

# df_ratings contient : userId, movieId, rating
# On va créer une matrice où :
# - les lignes = utilisateurs
# - les colonnes = films
# - les valeurs = ratings

user_item_matrix = df_ratings.pivot(index='userId', columns='movieId', values='rating')

# Afficher un aperçu
user_item_matrix


movieId,1,2,3,4,5,6,7,8,9,10,...,144482,144656,144976,146344,146656,146684,146878,148238,148626,149532
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5.0,NaN,2.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,3.0,NaN,3.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
665,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Collaborative Filtering

In [5]:
!pip install scikit-surprise


In [6]:
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split

# Définir le format des ratings
reader = Reader(rating_scale=(0.5, 5.0))

# Charger le DataFrame
data = Dataset.load_from_df(df_ratings[['userId', 'movieId', 'rating']], reader)

# Diviser en train/test
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)


In [42]:
from surprise import SVD
from surprise import accuracy

# Créer et entraîner le modèle
model = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02)
model.fit(trainset)

# Tester sur le testset
predictions = model.test(testset)

# Calculer RMSE
rmse = accuracy.rmse(predictions)
print("Test RMSE:", rmse)


RMSE: 0.8772
Test RMSE: 0.8772425054151258


In [36]:
def recommend_collaborative(user_id, top_n=10, df_movies=df_movies, df_ratings=df_ratings, model=model):
    # Films déjà notés
    rated_movies = df_ratings[df_ratings['userId'] == user_id]['movieId'].tolist()
    
    # Films à prédire
    movies_to_predict = df_movies[~df_movies['movieId'].isin(rated_movies)]
    
    # Prédictions
    predictions = []
    for movie in movies_to_predict['movieId']:
        pred = model.predict(user_id, movie)
        predictions.append((movie, pred.est))
    
    # Créer DataFrame pour conserver ordre et score
    pred_df = pd.DataFrame(predictions, columns=['movieId', 'pred_rating'])
    
    # Fusionner avec titres
    pred_df = pred_df.merge(df_movies[['movieId', 'title']], on='movieId', how='left')
    
    # Retourner top_n trié par note prédite
    return pred_df.sort_values('pred_rating', ascending=False).head(top_n)


## Content-Based Filtering

In [11]:
#Construire le profil utilisateur

#L’idée : représenter les goûts d’un utilisateur en genres.

def build_user_profile(user_id, df_ratings=df_ratings, df_movies=df_movies):
    # Récupérer les films notés par l'utilisateur
    user_ratings = df_ratings[df_ratings['userId'] == user_id].merge(df_movies, on='movieId')
    
    if user_ratings.empty:
        return None  # Nouvel utilisateur sans ratings
    
    # Colonnes des genres (Multi-Hot)
    genre_columns = [col for col in df_movies.columns if col not in ['movieId', 'title']]
    
    # Calculer la moyenne pondérée
    user_profile = (user_ratings[genre_columns].T @ user_ratings['rating']).T
    user_profile /= user_ratings['rating'].sum()
    
    return user_profile


In [37]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_content_based(user_id, top_n=10, df_movies=df_movies, df_ratings=df_ratings):
    profile = build_user_profile(user_id, df_ratings, df_movies)
    
    if profile is None:
        # Cold start complet → recommander films populaires ou au hasard
        return df_movies.head(top_n)[['movieId', 'title']]
    
    # Colonnes de genres
    genre_columns = [col for col in df_movies.columns if col not in ['movieId', 'title']]
    
    # Similarité cosine
    movie_vectors = df_movies[genre_columns]
    sim = cosine_similarity(profile.values.reshape(1, -1), movie_vectors)[0]
    
    # Copier le dataframe pour ajouter les scores
    df_movies_copy = df_movies.copy()
    df_movies_copy['score'] = sim
    
    # Exclure films déjà vus
    seen_movies = df_ratings[df_ratings['userId'] == user_id]['movieId'].tolist()
    
    # Retourner les top N films non vus triés par score
    return df_movies_copy[~df_movies_copy['movieId'].isin(seen_movies)].sort_values('score', ascending=False).head(top_n)


## Systéme hybride

In [38]:

def recommend_hybrid(user_id, top_n=10, df_movies=df_movies, df_ratings=df_ratings):
    n_ratings = df_ratings[df_ratings['userId'] == user_id].shape[0]

    if n_ratings == 0:
        # Cold start complet → Content-Based ou premiers films
        return recommend_content_based(user_id, top_n, df_movies, df_ratings)
    
    elif n_ratings < 10:
        # Peu de données → Content-Based
        return recommend_content_based(user_id, top_n, df_movies, df_ratings)
    
    else:
        # Utilisateur actif → SVD pur
        return recommend_collaborative(user_id, top_n, df_movies, df_ratings, model)


In [15]:
## TEST